In [3]:
"""
Enhanced Data Download for AI Crypto Trading Bot
Downloads comprehensive historical data for multiple cryptocurrencies across different timeframes.
"""
import ccxt
import pandas as pd
import numpy as np
import os
import sys
import time
import json
import logging
from datetime import datetime, timezone, timedelta
from typing import List, Dict, Optional
import warnings

# Add src to path for imports
notebook_dir = os.getcwd()  # Get the current working directory in Jupyter Notebook
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(notebook_dir))))

warnings.filterwarnings('ignore')

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('data_download.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Configuration
DATA_FOLDER = 'data' 
SYMBOLS = [
    'BTC/USDT', 'ETH/USDT', 'SOL/USDT', 'ADA/USDT', 
    'MATIC/USDT', 'DOT/USDT', 'LINK/USDT', 'AVAX/USDT',
    'UNI/USDT', 'LTC/USDT', 'ATOM/USDT', 'FTM/USDT'
]
TIMEFRAMES = ['6h', '12h', '1d', '3d']
LOOKBACK_YEARS = 3
BATCH_SIZE = 1000
RATE_LIMIT_DELAY = 0.1

class EnhancedDataDownloader:
    """Enhanced data downloader with robust error handling"""
    
    def __init__(self, exchange_name='binance', rate_limit=0.1):
        try:
            self.exchange = getattr(ccxt, exchange_name)({
                'options': {'defaultType': 'spot'},
                'enableRateLimit': True,
                'rateLimit': int(rate_limit * 1000)
            })
            self.rate_limit = rate_limit
            self.download_stats = {}
            logger.info(f"✅ Initialized {exchange_name} exchange")
        except Exception as e:
            logger.error(f"❌ Failed to initialize exchange: {e}")
            raise
    
    def validate_symbol(self, symbol: str) -> bool:
        """Validate if symbol exists and is active"""
        try:
            markets = self.exchange.load_markets()
            return symbol in markets and markets[symbol].get('active', False)
        except Exception as e:
            logger.error(f"Error validating symbol {symbol}: {e}")
            return False
    
    def get_timeframe_seconds(self, timeframe: str) -> int:
        """Convert timeframe string to seconds"""
        timeframe_map = {
            '1m': 60, '5m': 300, '15m': 900, '30m': 1800,
            '1h': 3600, '2h': 7200, '4h': 14400, '6h': 21600,
            '8h': 28800, '12h': 43200, '1d': 86400, '3d': 259200,
            '1w': 604800
        }
        return timeframe_map.get(timeframe, 3600)
    
    def validate_data_quality(self, df: pd.DataFrame, symbol: str, timeframe: str) -> Dict:
        """Validate downloaded data quality"""
        result = {'is_valid': True, 'errors': [], 'warnings': []}
        
        try:
            # Check minimum data length
            if len(df) < 100:
                result['is_valid'] = False
                result['errors'].append(f"Insufficient data: {len(df)} candles")
            
            # Check for null values
            null_counts = df.isnull().sum()
            if null_counts.any():
                result['warnings'].append(f"Null values found: {null_counts.to_dict()}")
            
            # Check price relationships
            invalid_highs = (df['high'] < df[['open', 'close']].max(axis=1)).sum()
            invalid_lows = (df['low'] > df[['open', 'close']].min(axis=1)).sum()
            
            if invalid_highs > 0:
                result['warnings'].append(f"Invalid high prices: {invalid_highs} candles")
            if invalid_lows > 0:
                result['warnings'].append(f"Invalid low prices: {invalid_lows} candles")
            
            # Check for reasonable price ranges
            price_cols = ['open', 'high', 'low', 'close']
            if (df[price_cols] <= 0).any().any():
                result['is_valid'] = False
                result['errors'].append("Zero or negative prices found")
            
            # Check for extreme price changes (>50% in one candle)
            price_changes = df['close'].pct_change().abs()
            extreme_changes = (price_changes > 0.5).sum()
            if extreme_changes > 5:
                result['warnings'].append(f"Many extreme price changes: {extreme_changes}")
            
            # Check time consistency
            if len(df) > 1:
                time_diffs = df['timestamp'].diff().dt.total_seconds()
                expected_interval = self.get_timeframe_seconds(timeframe)
                irregular_intervals = ((time_diffs < expected_interval * 0.8) | 
                                     (time_diffs > expected_interval * 1.2)).sum()
                
                if irregular_intervals > len(df) * 0.1:
                    result['warnings'].append(f"Many irregular time intervals: {irregular_intervals}")
            
        except Exception as e:
            result['is_valid'] = False
            result['errors'].append(f"Validation error: {str(e)}")
        
        return result
    
    def download_with_pagination(self, symbol: str, timeframe: str, 
                                start_date: datetime, end_date: datetime) -> Optional[pd.DataFrame]:
        """Download data with automatic pagination"""
        try:
            logger.info(f"Downloading {symbol} {timeframe} from {start_date} to {end_date}")
            
            all_ohlcv = []
            since = int(start_date.timestamp() * 1000)
            until = int(end_date.timestamp() * 1000)
            
            request_count = 0
            max_requests = 1000
            
            while since < until and request_count < max_requests:
                try:
                    # Rate limiting
                    time.sleep(self.rate_limit)
                    
                    # Fetch batch
                    ohlcv = self.exchange.fetch_ohlcv(
                        symbol, timeframe=timeframe, since=since, limit=BATCH_SIZE
                    )
                    
                    if not ohlcv:
                        break
                    
                    # Filter candles within our timeframe
                    filtered_ohlcv = [
                        candle for candle in ohlcv 
                        if since <= candle[0] <= until
                    ]
                    
                    all_ohlcv.extend(filtered_ohlcv)
                    
                    # Update since for next batch
                    since = ohlcv[-1][0] + 1
                    request_count += 1
                    
                    if request_count % 10 == 0:
                        logger.info(f"  Progress: {len(all_ohlcv)} candles, {request_count} requests")
                
                except ccxt.RateLimitExceeded:
                    logger.warning(f"Rate limit exceeded for {symbol}, waiting 5 seconds...")
                    time.sleep(5)
                except ccxt.NetworkError as e:
                    logger.warning(f"Network error for {symbol}: {e}, retrying...")
                    time.sleep(2)
                except Exception as e:
                    logger.error(f"Unexpected error for {symbol}: {e}")
                    break
            
            if all_ohlcv:
                # Create DataFrame
                df = pd.DataFrame(
                    all_ohlcv, 
                    columns=['timestamp', 'open', 'high', 'low', 'close', 'volume']
                )
                
                # Convert timestamp and remove duplicates
                df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
                df = df.drop_duplicates(subset='timestamp', keep='first')
                df = df.sort_values('timestamp').reset_index(drop=True)
                
                # Validate data quality
                validation_result = self.validate_data_quality(df, symbol, timeframe)
                
                if validation_result['is_valid']:
                    logger.info(f"✅ Downloaded {len(df)} candles for {symbol} {timeframe}")
                    return df
                else:
                    logger.error(f"❌ Data validation failed for {symbol}: {validation_result['errors']}")
                    return None
            else:
                logger.warning(f"No data downloaded for {symbol} {timeframe}")
                return None
                
        except Exception as e:
            logger.error(f"Failed to download {symbol} {timeframe}: {e}")
            return None
    
    def save_data(self, df: pd.DataFrame, symbol: str, timeframe: str) -> Optional[str]:
        """Save data to CSV file"""
        try:
            # Ensure data folder exists
            os.makedirs(DATA_FOLDER, exist_ok=True)
            
            # Create filename
            symbol_clean = symbol.replace('/', '_')
            filename = f"{symbol_clean}_{timeframe}.csv"
            filepath = os.path.join(DATA_FOLDER, filename)
            
            # Save to CSV
            df.to_csv(filepath, index=False)
            
            # Update stats
            self.download_stats[f"{symbol}_{timeframe}"] = {
                'candles': len(df),
                'start_date': df['timestamp'].iloc[0].isoformat(),
                'end_date': df['timestamp'].iloc[-1].isoformat(),
                'file_path': filepath,
                'file_size_mb': os.path.getsize(filepath) / 1024 / 1024
            }
            
            logger.info(f"✅ Saved {len(df)} candles to {filepath}")
            return filepath
            
        except Exception as e:
            logger.error(f"Failed to save data for {symbol} {timeframe}: {e}")
            return None
     
    def download_all_data(self, symbols: List[str], timeframes: List[str], 
                         lookback_years: int = 3) -> Dict:
        """Download data for all symbols and timeframes"""
        start_date = datetime.now(timezone.utc) - timedelta(days=lookback_years * 365)
        end_date = datetime.now(timezone.utc)
        
        results = {
            'successful': [],
            'failed': [],
            'stats': {}
        }
        
        total_combinations = len(symbols) * len(timeframes)
        current_combination = 0
        
        logger.info(f"🚀 Starting download of {total_combinations} combinations")
        logger.info(f"Date range: {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")
        
        for symbol in symbols:
            # Validate symbol first
            if not self.validate_symbol(symbol):
                logger.error(f"❌ Invalid symbol: {symbol}")
                for tf in timeframes:
                    results['failed'].append(f"{symbol}_{tf}")
                    current_combination += 1
                continue
            
            for timeframe in timeframes:
                current_combination += 1
                progress = (current_combination / total_combinations) * 100
                
                logger.info(f"\n{'='*50}")
                logger.info(f"Progress: {progress:.1f}% ({current_combination}/{total_combinations})")
                logger.info(f"Downloading: {symbol} {timeframe}")
                
                try:
                    # Download data
                    df = self.download_with_pagination(symbol, timeframe, start_date, end_date)
                    
                    if df is not None and not df.empty:
                        # Save data
                        filepath = self.save_data(df, symbol, timeframe)
                        
                        if filepath:
                            results['successful'].append(f"{symbol}_{timeframe}")
                        else:
                            results['failed'].append(f"{symbol}_{timeframe}")
                    else:
                        results['failed'].append(f"{symbol}_{timeframe}")
                        
                except Exception as e:
                    logger.error(f"❌ Failed {symbol} {timeframe}: {e}")
                    results['failed'].append(f"{symbol}_{timeframe}")
                
                # Small delay between downloads
                time.sleep(1)
        
        results['stats'] = self.download_stats
        return results

def create_data_quality_report(results: Dict):
    """Create comprehensive data quality report"""
    logger.info(f"\n📊 DATA QUALITY REPORT")
    logger.info(f"{'='*60}")
    
    total_candles = 0
    file_sizes = []
    
    for key, stats in results['stats'].items():
        candles = stats['candles']
        start = stats['start_date']
        end = stats['end_date']
        file_size = stats['file_size_mb']
        
        total_candles += candles
        file_sizes.append(file_size)
        
        logger.info(f"{key}: {candles:,} candles ({start[:10]} to {end[:10]}) - {file_size:.2f}MB")
    
    logger.info(f"\n📈 SUMMARY:")
    logger.info(f"Total candles downloaded: {total_candles:,}")
    logger.info(f"Total data size: {sum(file_sizes):.2f}MB")
    logger.info(f"Average file size: {np.mean(file_sizes):.2f}MB")
    logger.info(f"Files created: {len(file_sizes)}")

def verify_data_integrity(results: Dict):
    """Verify data integrity"""
    logger.info(f"\n🔍 DATA INTEGRITY CHECK")
    logger.info(f"{'='*60}")
    
    integrity_issues = []
    
    for key, stats in results['stats'].items():
        filepath = stats['file_path']
        
        try:
            # Load and check data
            df = pd.read_csv(filepath, parse_dates=['timestamp'])
            
            # Check for issues
            issues = []
            
            # Check for gaps in time series
            df_sorted = df.sort_values('timestamp')
            if len(df_sorted) > 1:
                time_diffs = df_sorted['timestamp'].diff()
                
                # Expected interval based on timeframe
                timeframe = key.split('_')[-1]
                downloader = EnhancedDataDownloader()
                expected_interval = downloader.get_timeframe_seconds(timeframe)
                
                large_gaps = time_diffs > pd.Timedelta(seconds=expected_interval * 2)
                gap_count = large_gaps.sum()
                
                if gap_count > 0:
                    issues.append(f"{gap_count} large time gaps")
            
            # Check for duplicate timestamps
            duplicates = df['timestamp'].duplicated().sum()
            if duplicates > 0:
                issues.append(f"{duplicates} duplicate timestamps")
            
            # Check for null values
            null_count = df.isnull().sum().sum()
            if null_count > 0:
                issues.append(f"{null_count} null values")
            
            if issues:
                integrity_issues.append(f"{key}: {', '.join(issues)}")
                logger.info(f"⚠️ {key}: {', '.join(issues)}")
            else:
                logger.info(f"✅ {key}: No issues found")
                
        except Exception as e:
            integrity_issues.append(f"{key}: Error reading file - {e}")
            logger.error(f"❌ {key}: Error reading file - {e}")
    
    if not integrity_issues:
        logger.info(f"\n🎉 All data files passed integrity checks!")
    else:
        logger.info(f"\n⚠️ Found {len(integrity_issues)} files with issues that may need attention.")
    
    return integrity_issues

def save_download_summary(results: Dict, integrity_issues: List):
    """Save download summary report"""
    total_candles = sum(stats['candles'] for stats in results['stats'].values())
    file_sizes = [stats['file_size_mb'] for stats in results['stats'].values()]
    
    summary_report = {
        'download_timestamp': datetime.now(timezone.utc).isoformat(),
        'symbols': SYMBOLS,
        'timeframes': TIMEFRAMES,
        'lookback_years': LOOKBACK_YEARS,
        'total_combinations': len(SYMBOLS) * len(TIMEFRAMES),
        'successful_downloads': len(results['successful']),
        'failed_downloads': len(results['failed']),
        'total_candles': total_candles,
        'total_size_mb': sum(file_sizes) if file_sizes else 0,
        'successful_files': results['successful'],
        'failed_files': results['failed'],
        'integrity_issues': integrity_issues,
        'detailed_stats': results['stats']
    }
    
    # Save summary
    os.makedirs(DATA_FOLDER, exist_ok=True)
    summary_path = os.path.join(DATA_FOLDER, 'download_summary.json')
    with open(summary_path, 'w') as f:
        json.dump(summary_report, f, indent=2, default=str)
    
    logger.info(f"\n📋 Download summary saved to: {summary_path}")
    return summary_path

def main():
    """Main function to run the enhanced data download"""
    logger.info("🤖 AI CRYPTO TRADING BOT - ENHANCED DATA DOWNLOAD")
    logger.info(f"🕐 {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')} UTC")
    logger.info(f"👤 User: samannazir55")
    logger.info("=" * 60)
    
    try:
        # Initialize downloader
        downloader = EnhancedDataDownloader('binance', RATE_LIMIT_DELAY)
        logger.info("✅ Enhanced Data Downloader initialized")
        
        # Show configuration
        logger.info(f"\nConfiguration:")
        logger.info(f"Symbols: {SYMBOLS}")
        logger.info(f"Timeframes: {TIMEFRAMES}")
        logger.info(f"Lookback: {LOOKBACK_YEARS} years")
        logger.info(f"Total combinations: {len(SYMBOLS) * len(TIMEFRAMES)}")
        
        # Run download
        results = downloader.download_all_data(SYMBOLS, TIMEFRAMES, LOOKBACK_YEARS)
        
        # Show results
        logger.info(f"\n{'='*60}")
        logger.info(f"📊 DOWNLOAD COMPLETE")
        logger.info(f"✅ Successful: {len(results['successful'])}")
        logger.info(f"❌ Failed: {len(results['failed'])}")
        
        if results['failed']:
            logger.info(f"\nFailed downloads:")
            for failed in results['failed']:
                logger.info(f"  - {failed}")
        
        # Create reports
        if results['successful']:
            create_data_quality_report(results)
            integrity_issues = verify_data_integrity(results)
            summary_path = save_download_summary(results, integrity_issues)
            
            logger.info(f"\n✅ Data download completed successfully!")
            logger.info(f"📋 Summary saved to: {summary_path}")
            logger.info(f"📁 Data files saved to: {DATA_FOLDER}/")
            logger.info(f"\nNext step: Run 02_feature_extraction.py")
        else:
            logger.error(f"\n❌ No data was successfully downloaded!")
            return False
        
        return True
        
    except Exception as e:
        logger.error(f"❌ Data download failed: {e}")
        return False

if __name__ == "__main__":
    success = main()
    if not success:
        exit(1)

--- Logging error ---
Traceback (most recent call last):
  File "C:\ProgramData\anaconda3\Lib\logging\__init__.py", line 1154, in emit
    stream.write(msg + self.terminator)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\ProgramData\anaconda3\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\U0001f916' in position 33: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "d:\CryptoSight\venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "d:\CryptoSight\venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "d:\CryptoSight\venv\Lib\site-packages\ipykernel\kernelapp.py", line 739, in star